In [1]:
run_on_colab = False
if run_on_colab:
    !pip install tiktoken
    %rm -rf aptax
    !git clone https://github.com/antodima/aptax.git
    %cd aptax/
    # !pip install uv
    # !uv pip freeze > requirements.txt
else:
    %cd ..

/Users/antodima/Code/aptax


In [2]:
import tiktoken
from collections import defaultdict
from tqdm import tqdm
from pathlib import Path

import jax
import jax.numpy as jnp
from jax.sharding import SingleDeviceSharding 

import flax.nnx as nnx
from flax.training.early_stopping import EarlyStopping

import optax
import orbax
from orbax import checkpoint

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from aptax.llm import MiniGPT
from aptax.dataset import create_dataloader
from aptax.dataset import load_stories, TextsDataset

In [3]:
tokenizer = tiktoken.get_encoding("gpt2")
vocab_size = tokenizer.n_vocab

max_seq_len = 1024
embed_dim = 192
num_heads = 6
batch_size = 32
num_transformer_blocks = 6
num_epochs = 300

texts = load_stories()

dataset = TextsDataset(texts, max_seq_len, tokenizer)
dataloader, batches_per_epoch = create_dataloader(dataset, batch_size=batch_size, num_epochs=1)
total_steps = num_epochs * batches_per_epoch
warmup_steps = max(1, total_steps // 10)
print(f"Dataset size: {len(dataset)}")
print(f"Batches per epoch: {batches_per_epoch}")
print(f"Total training steps: {total_steps:,}")
print(f"Warmup steps: {warmup_steps:,}")

Loading stories from aptax/data/tinystories/TinyStories-1000.txt...
Loaded 1,000 stories
Dataset size: 1000
Batches per epoch: 31
Total training steps: 9,300
Warmup steps: 930


In [4]:
model = MiniGPT(
    max_seq_len=max_seq_len,
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    num_heads=num_heads,
    num_transformer_blocks=num_transformer_blocks,
    rngs=nnx.Rngs(0),
)
print(model)

W0313 21:17:38.198945 1801410 cpp_gen_intrinsics.cc:74] Empty bitcode string provided for eigen. Optimizations relying on this IR will be disabled.


MiniGPT( # RngState: 2 (12 B), Param: 20,384,640 (81.5 MB), Total: 20,384,642 (81.5 MB)
  max_seq_len=1024,
  embedding=TokenAndPositionEmbedding( # Param: 9,845,952 (39.4 MB)
    token_emb=Embed( # Param: 9,649,344 (38.6 MB)
      embedding=Param( # 9,649,344 (38.6 MB)
        value=Array(shape=(50257, 192), dtype=dtype('float32'))
      ),
      num_embeddings=50257,
      features=192,
      dtype=dtype('float32'),
      param_dtype=float32,
      promote_dtype=<function promote_dtype at 0x1144662a0>
    ),
    pos_emb=Embed( # Param: 196,608 (786.4 KB)
      embedding=Param( # 196,608 (786.4 KB)
        value=Array(shape=(1024, 192), dtype=dtype('float32'))
      ),
      num_embeddings=1024,
      features=192,
      dtype=dtype('float32'),
      param_dtype=float32,
      promote_dtype=<function promote_dtype at 0x1144662a0>
    )
  ),
  transformer_blocks=[TransformerBlock( # Param: 148,224 (592.9 KB)
    attn=MultiHeadAttention( # Param: 148,224 (592.9 KB)
      num_heads=6,
  

In [ ]:
lr_schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=5e-4,
    warmup_steps=warmup_steps,
    decay_steps=total_steps,
    end_value=5e-5,
)
optimizer = nnx.ModelAndOptimizer(
    model,
    optax.adamw(learning_rate=lr_schedule, weight_decay=0.01),
)
metrics = nnx.MultiMetric(
    loss=nnx.metrics.Average('loss'),
    accuracy=nnx.metrics.Average('accuracy'),
)

In [ ]:
def loss_fn(model, batch):
    inputs, targets = batch
    logits = model(inputs)
    loss = optax.softmax_cross_entropy_with_integer_labels(
        logits, targets
    ).mean()
    return loss, logits
    

@nnx.jit
def train_step(model, optimizer, metrics, batch):
    inputs, targets = batch
    grad_fn = nnx.value_and_grad(loss_fn, has_aux=True)
    (loss, logits), grads = grad_fn(model, batch)
    accuracy = (logits.argmax(axis=-1) == targets).mean()

    metrics.update(loss=loss, accuracy=accuracy, logits=logits, labels=targets)
    optimizer.update(grads)

def eval_step(model, metrics, batch):
    inputs, targets = batch
    loss, logits = loss_fn(model, batch)
    accuracy = (logits.argmax(axis=-1) == targets).mean()
    metrics.update(loss=loss, accuracy=accuracy, logits=logits, labels=targets)

In [ ]:
prep_target_batch = jax.vmap(
    lambda tokens: jnp.concatenate((tokens[1:], jnp.array([dataset.eot_token_id]))))

step = 0
score_history = defaultdict(list)
early_stop = EarlyStopping(min_delta=1e-2, patience=5)
for epoch in range(num_epochs):
    epoch_history = defaultdict(list)
    train_batches, test_batches = train_test_split(list(range(batches_per_epoch)), test_size=0.2, random_state=epoch)
    batch_idx = 0
    for batch in (pbar:=tqdm(dataloader, total=batches_per_epoch)):
        inputs = jnp.array(batch).T
        targets = prep_target_batch(inputs)

        step_type = "train" if batch_idx in train_batches else "valid"
        if step_type == "train":
            train_step(model, optimizer, metrics, (inputs, targets))
        else:
            eval_step(model, metrics, (inputs, targets))

        for metric, value in metrics.compute().items():
            epoch_history[f"{step_type}_{metric}"].append(value)
        metrics.reset()

        current_lr = lr_schedule(step)
        pbar.set_description(
            f"epoch: {epoch+1}/{num_epochs}, lr: {current_lr:.2e}, "
            f"loss (train/valid): {np.mean(epoch_history['train_loss']):.3f}/{np.mean(epoch_history['valid_loss']):.3f}, "
            f"acc (train/valid): {np.mean(epoch_history['train_accuracy']):.3f}/{np.mean(epoch_history['valid_accuracy']):.3f}"
        )
        step += 1
        batch_idx += 1

    for metric, values in epoch_history.items():
        score_history[metric].append(values)
    early_stop = early_stop.update(np.mean(score_history['train_loss']))
    if early_stop.should_stop:
        print(f"\nMet early stopping criteria! Breaking at epoch {epoch}.")
        break

In [ ]:
train_loss = np.array(score_history['train_loss'])
valid_loss = np.array(score_history['valid_loss'])

plt.plot(train_loss.mean(axis=1), label='train')
plt.fill_between(
    np.arange(0, len(train_loss)), 
    np.subtract(train_loss.mean(axis=1), train_loss.std(axis=1)), 
    np.add(train_loss.mean(axis=1), train_loss.std(axis=1)), 
    alpha=0.2, 
    linestyle='-',
)
plt.plot(valid_loss.mean(axis=1), label='valid')
plt.fill_between(
    np.arange(0, len(valid_loss)), 
    np.subtract(valid_loss.mean(axis=1), valid_loss.std(axis=1)), 
    np.add(valid_loss.mean(axis=1), valid_loss.std(axis=1)), 
    alpha=0.2, 
    linestyle='-',
)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

In [ ]:
checkpoint_path = Path.cwd() / "checkpoints" / "minigpt.orbax"
checkpointer = orbax.checkpoint.PyTreeCheckpointer()
checkpointer.save(checkpoint_path, nnx.state(model), force=True)
print(f"Model saved as {checkpoint_path}")
if run_on_colab:
    !zip -r minigpt.orbax.zip $checkpoint_path

In [5]:
checkpoint_path = Path.cwd() / "checkpoints" / "minigpt.orbax"
checkpointer = orbax.checkpoint.PyTreeCheckpointer()

cpu_device = jax.devices('cpu')[0]
cpu_sharding = SingleDeviceSharding(cpu_device)
restore_args = jax.tree_util.tree_map(
    lambda _: checkpoint.ArrayRestoreArgs(sharding=cpu_sharding),
    nnx.state(model)
)
nnx.state(model)
restored_state = checkpointer.restore(
    checkpoint_path,
    item=nnx.state(model),
    restore_args=restore_args)

nnx.update(model, restored_state)

In [6]:
def generate_text(model, start_tokens, max_new_tokens=50, temperature=1.0):
    tokens = list(start_tokens)
    for _ in range(max_new_tokens):
        context = tokens[-model.max_seq_len:]
        # RIGHT-pad to match training (not left-pad!)
        actual_len = len(context)
        if actual_len < model.max_seq_len:
            context = context + [0] * (model.max_seq_len - actual_len)

        context_array = jnp.atleast_2d(jnp.array(context))
        logits = model(context_array)
        next_token_logits = logits[0, actual_len - 1, :] / temperature
        next_token = int(jnp.argmax(next_token_logits))
        if next_token == tokenizer.encode(dataset.eot_token, allowed_special={dataset.eot_token})[0]:
            break
        tokens.append(next_token)
    return tokenizer.decode(tokens)

def generate_story(model, story_prompt, temperature, max_new_tokens):
    start_tokens = tokenizer.encode(story_prompt)[:model.max_seq_len]
    generated = generate_text(model, start_tokens, max_new_tokens=max_new_tokens, temperature=temperature)
    return generated

def create_story(story_prompt, temperature, max_new_tokens):
    return generate_story(model, story_prompt, temperature, max_new_tokens)

create_story("Once upon a time", 0.2, 50)

'Once upon a time, there was a little girl named Lily. She loved to play outside in the park with her family. One day, she found a loud sound, "Lily, sweetie, so pretty. Can I wanted to play with the slide. '